In [1]:
import tkinter as tk
import math
import heapq

# ---------------- CONFIG ----------------
COLORS = {
    "bg": "#1a1b26",
    "sidebar": "#24283b",
    "node": "#7aa2f7",
    "edge": "#414868",
    "path": "#ff9e64",
    "text": "#c0caf5",
    "start": "#9ece6a",
    "end": "#f7768e",
    "detail_bg": "#16161e"
}

coords = {
    "Agrabad": (100, 300), "Dewanhat": (150, 250), "Tigerpass": (200, 200),
    "Lalkhan Bazar": (250, 180), "Kotwali": (300, 160), "Anderkilla": (350, 160),
    "Chawkbazar": (400, 160), "New Market": (450, 160), "Panchlaish": (400, 200),
    "GEC": (300, 220), "Nasirabad": (250, 250), "Foy's Lake": (200, 280),
    "Oxygen": (150, 320), "2 No Gate": (350, 240), "Muradpur": (400, 280),
    "Bahaddarhat": (450, 320), "Kalurghat": (500, 350),
    "Halishahar": (80, 260), "EPZ": (50, 300), "Patenga": (20, 350)
}

graph = {
    "Agrabad": {"Dewanhat": 2, "Halishahar": 4, "EPZ": 6},
    "Dewanhat": {"Agrabad": 2, "Tigerpass": 3},
    "Tigerpass": {"Dewanhat": 3, "Lalkhan Bazar": 2, "GEC": 3},
    "Lalkhan Bazar": {"Tigerpass": 2, "Kotwali": 2},
    "Kotwali": {"Lalkhan Bazar": 2, "Anderkilla": 1},
    "Anderkilla": {"Kotwali": 1, "Chawkbazar": 1},
    "Chawkbazar": {"Anderkilla": 1, "New Market": 1},
    "New Market": {"Chawkbazar": 1, "Panchlaish": 2},
    "Panchlaish": {"New Market": 2, "GEC": 2, "2 No Gate": 3},
    "GEC": {"Tigerpass": 3, "Panchlaish": 2, "Nasirabad": 2},
    "Nasirabad": {"GEC": 2, "Foy's Lake": 2},
    "Foy's Lake": {"Nasirabad": 2, "Oxygen": 2},
    "Oxygen": {"Foy's Lake": 2, "Agrabad": 3},
    "2 No Gate": {"Panchlaish": 3, "Muradpur": 3},
    "Muradpur": {"2 No Gate": 3, "Bahaddarhat": 3},
    "Bahaddarhat": {"Muradpur": 3, "Kalurghat": 4},
    "Kalurghat": {"Bahaddarhat": 4},
    "Halishahar": {"Agrabad": 4, "EPZ": 3},
    "EPZ": {"Agrabad": 6, "Halishahar": 3, "Patenga": 5},
    "Patenga": {"EPZ": 5}
}

def heuristic(a, b):
    x1, y1 = coords[a]; x2, y2 = coords[b]
    return math.hypot(x1 - x2, y1 - y2)

def astar(start, goal):
    open_list = [(0, start)]
    came = {}
    g = {n: float('inf') for n in graph}; g[start] = 0
    while open_list:
        _, curr = heapq.heappop(open_list)
        if curr == goal:
            path = []
            total_cost = g[goal]
            while curr in came:
                path.append(curr); curr = came[curr]
            path.append(start)
            return path[::-1], total_cost
        for nb, weight in graph[curr].items():
            temp_g = g[curr] + weight
            if temp_g < g[nb]:
                came[nb] = curr
                g[nb] = temp_g
                heapq.heappush(open_list, (temp_g + heuristic(nb, goal), nb))
    return [], 0

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("🚚 Smart Logistics Map")
        self.root.geometry("950x550")
        self.root.configure(bg=COLORS["bg"])
        
        self.selected = []
        
        # Sidebar
        self.sidebar = tk.Frame(root, bg=COLORS["sidebar"], width=250)
        self.sidebar.pack(side="left", fill="y")
        self.sidebar.pack_propagate(False) # Keep width fixed
        
        tk.Label(self.sidebar, text="ROUTE PLANNER", fg=COLORS["path"], bg=COLORS["sidebar"], font=("Arial", 14, "bold")).pack(pady=20)
        
        self.info = tk.Label(self.sidebar, text="📍 Click Start Location", fg=COLORS["text"], bg=COLORS["sidebar"], font=("Arial", 10))
        self.info.pack(pady=5)

        # Route Detail Area
        tk.Label(self.sidebar, text="PATH STEPS:", fg=COLORS["node"], bg=COLORS["sidebar"], font=("Arial", 9, "bold")).pack(anchor="w", padx=20, pady=(20,5))
        self.detail_frame = tk.Frame(self.sidebar, bg=COLORS["detail_bg"], padx=10, pady=10)
        self.detail_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        self.detail_text = tk.Label(self.detail_frame, text="", fg=COLORS["text"], bg=COLORS["detail_bg"], justify="left", anchor="nw", wraplength=200, font=("Consolas", 9))
        self.detail_text.pack(fill="both", expand=True)
        
        self.cost_lbl = tk.Label(self.sidebar, text="Total Distance: 0 km", fg=COLORS["start"], bg=COLORS["sidebar"], font=("Arial", 10, "bold"))
        self.cost_lbl.pack(pady=10)
        
        tk.Button(self.sidebar, text="Reset Map", command=self.reset, bg="#3b4261", fg="white", relief="flat", padx=20).pack(side="bottom", pady=20)

        # Canvas
        self.canvas = tk.Canvas(root, bg=COLORS["bg"], highlightthickness=0)
        self.canvas.pack(side="right", fill="both", expand=True)
        
        self.draw_map()
        self.canvas.bind("<Button-1>", self.on_click)

    def draw_map(self):
        self.canvas.delete("all")
        for a, nbs in graph.items():
            for b in nbs:
                x1, y1 = coords[a]; x2, y2 = coords[b]
                self.canvas.create_line(x1, y1, x2, y2, fill=COLORS["edge"], width=2)
        
        for city, (x, y) in coords.items():
            self.canvas.create_oval(x-5, y-5, x+5, y+5, fill=COLORS["node"], outline="white", tags=city)
            self.canvas.create_text(x, y-15, text=city, fill=COLORS["text"], font=("Arial", 8))

    def on_click(self, event):
        city = min(coords, key=lambda p: (coords[p][0]-event.x)**2 + (coords[p][1]-event.y)**2)
        
        if len(self.selected) < 2:
            self.selected.append(city)
            color = COLORS["start"] if len(self.selected) == 1 else COLORS["end"]
            self.canvas.create_oval(coords[city][0]-8, coords[city][1]-8, coords[city][0]+8, coords[city][1]+8, 
                                    outline=color, width=3, tags="marker")
            
            if len(self.selected) == 1:
                self.info.config(text=f"Start: {city}")
                self.detail_text.config(text=f"1. {city}\n   (Waiting for goal...)")
            else:
                self.info.config(text=f"Goal Reached!")
                self.draw_path()

    def draw_path(self):
        path, cost = astar(self.selected[0], self.selected[1])
        if not path:
            self.detail_text.config(text="❌ No Path Found!")
            return

        # Update Sidebar Text with Step-by-Step
        path_string = ""
        for i, step in enumerate(path):
            path_string += f"{i+1}. {step}\n"
        
        self.detail_text.config(text=path_string)
        self.cost_lbl.config(text=f"Total Distance: {cost} km")
        
        # Animate Path
        for i in range(len(path)-1):
            x1, y1 = coords[path[i]]
            x2, y2 = coords[path[i+1]]
            self.canvas.create_line(x1, y1, x2, y2, fill=COLORS["path"], width=4, tags="path_line")
            self.root.update()
            self.root.after(100)

    def reset(self):
        self.selected = []
        self.info.config(text="📍 Click Start Location")
        self.detail_text.config(text="")
        self.cost_lbl.config(text="Total Distance: 0 km")
        self.draw_map()

if __name__ == "__main__":
    root = tk.Tk()
    App(root)
    root.mainloop()